# Validation — `vqasynth-pp-spatial-quality`

**Target metric:** `metric_answer_within_bar_rate`

**Repository:** [remyxai/VQASynth](https://github.com/remyxai/VQASynth) at commit [`d64156f8f65c`](https://github.com/remyxai/VQASynth/commit/d64156f8f65c3eed980818d2897ef42687da90d9)

**Benchmark:** the repository's own `` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Verdict: `no-signal`** · Remyx run `2dcb3db9`

The outputs below are the ones that run produced on Remyx compute. Run the cells top to bottom on a machine with a GPU to reproduce them.

In [ ]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

## 1. Environment

On Remyx compute this ran in `pytorch/pytorch:2.14.0-cuda12.6-cudnn9-runtime`; the exact Dockerfile is committed at `.remyx/validations/vqasynth-pp-spatial-quality.Dockerfile` — edit it and the next run builds from your version.

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [ ]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/remyxai/VQASynth"
COMMIT = ref or "d64156f8f65c3eed980818d2897ef42687da90d9"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

## 4. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [ ]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

## 5. Run the benchmark

``

In [ ]:
os.chdir(ROOT)
import glob, importlib, runpy, sys, time
RUN_STARTED = time.time()
sys.argv = ["run.py"]
runpy.run_path("run.py", run_name="__main__")

/bin/sh: 1: eval/eval_vqasynth_spatial_quality.py: not found

## 6. Read what the benchmark wrote

Results land under `results/*.json` — or wherever this harness writes for a non-default checkout; only a document written by the run above counts. The metrics the criteria are judged against are fields of that document.

In [ ]:
import glob, json, os
PATTERNS = ["results/*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
paths = [p for p in paths if os.path.getmtime(p) >= RUN_STARTED - 1]
if not paths:
    # Some harnesses write elsewhere depending on the checkout (peft uses
    # temporary_results/ off the main branch): any document this run wrote.
    paths = sorted((p for p in glob.glob("**/*.json", recursive=True)
                    if os.path.getmtime(p) >= RUN_STARTED - 1 and "experiments/" not in p),
                   key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = ["metric_answer_within_bar_rate", "localize_roundtrip_rate", "perspective_flip_accuracy", "metric_answer_rel_error", "retired_x10_rel_error", "grounded_label_coverage", "inlier_ratio_mean", "unusable_extrinsics_accept_rate", "real_registration_spread_ratio", "real_object_merge_rate", "real_depth_conf_variation_rate", "real_scale_sanity_rate", "real_distance_plausibility_rate", "s1_message_hash_crc32", "s1_message_count", "fused_new_family_message_count"]
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

(the benchmark wrote no result document — see the run above)


## 7. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [ ]:
CRITERIA = [
    {
        "metric": "metric_answer_within_bar_rate",
        "direction": ">=",
        "threshold": 1.0,
        "baseline": null
    },
    {
        "metric": "localize_roundtrip_rate",
        "direction": ">=",
        "threshold": 1.0,
        "baseline": null
    },
    {
        "metric": "perspective_flip_accuracy",
        "direction": ">=",
        "threshold": 1.0,
        "baseline": null
    },
    {
        "metric": "metric_answer_rel_error",
        "direction": "<=",
        "threshold": 0.05,
        "baseline": null
    },
    {
        "metric": "retired_x10_rel_error",
        "direction": ">=",
        "threshold": 0.5,
        "baseline": null
    },
    {
        "metric": "grounded_label_coverage",
        "direction": ">=",
        "threshold": 0.66,
        "baseline": null
    },
    {
        "metric": "inlier_ratio_mean",
        "direction": ">=",
        "threshold": 0.65,
        "baseline": null
    },
    {
        "metric": "unusable_extrinsics_accept_rate",
        "direction": "<=",
        "threshold": 0.0,
        "baseline": null
    },
    {
        "metric": "real_registration_spread_ratio",
        "direction": "<=",
        "threshold": 0.05,
        "baseline": null
    },
    {
        "metric": "real_object_merge_rate",
        "direction": ">=",
        "threshold": 1.0,
        "baseline": null
    },
    {
        "metric": "real_depth_conf_variation_rate",
        "direction": ">=",
        "threshold": 1.0,
        "baseline": null
    },
    {
        "metric": "real_scale_sanity_rate",
        "direction": ">=",
        "threshold": 1.0,
        "baseline": null
    },
    {
        "metric": "real_distance_plausibility_rate",
        "direction": ">=",
        "threshold": 1.0,
        "baseline": null
    },
    {
        "metric": "s1_message_hash_crc32",
        "direction": ">=",
        "threshold": 0,
        "baseline": null
    },
    {
        "metric": "s1_message_count",
        "direction": ">=",
        "threshold": 1,
        "baseline": null
    },
    {
        "metric": "fused_new_family_message_count",
        "direction": ">=",
        "threshold": 1,
        "baseline": null
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    fmt = lambda x: (f"{x:.6g}" if isinstance(x, float) else str(x))
    print(f"{c['metric']:<28}{fmt(v):>16}{fmt(c['baseline']):>16}  {c['direction']} {fmt(t)}  {mark}")

## 8. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [ ]:
print(json.dumps(observed))

## 9. What the outcome means

**Verdict: `no-signal`.** Every change the validation loop made on the way is a separate commit on this branch, named for the run that applied it — revert any of them to undo it.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
question:
  kind: compare
  ask: "Keep the CPU-observable bars but add a real-GPU arm running the actual VGGT + DepthPro paths unstubbed, plus a legacy-S=1 vs PR two-arm comparison on identical real clips with S=1 message-hash parity - does VQASynth++ Phase 1-2 deliver real-world-calibrated, viewpoint-aware spatial data with the single-image path preserved?"
benchmarks:
  - name: vqasynth-pp-spatial-quality
    suite: "eval/eval_vqasynth_spatial_quality.py"
    packages: []  # Python wheels ONLY - libegl1 and every other lib*/.so OS library is an
    # apt package pip can never resolve (user_guidance), so none appears here; the
    # open3d headless runtime set (libgl1 libegl1 libgomp1 libglib2.0-0) is installed
    # system-side by the script's apt-install prologue. No PyPI distribution named
    # vggt exists either: the unstubbed import is provided by the repo's own
    # environment (PR-head dependency set) and its absence is handled by the
    # script's declared ImportError fallback.
    scorer: metric_answer_within_bar_rate
    baseline: main
    policy:
      guardrail_veto: true
    metrics:
      - name: metric_answer_within_bar_rate
        direction: max
        threshold: 1.0
        role: target
        bar: goal
        reads_as: "Share of emitted distance/size answers matching known ground-truth meters within the PR bar (rel_tol=0.05 or half-unit rounding)"
        # Threshold derivation: the calibrated path is exact pinhole unprojection on a
        # synthetic scene (focal 100px, depth 5 m -> 0.05 m/px), so answers move only by
        # the humanizer's unit rounding, which the bar's half-unit branch absorbs
        # (worst case 0.4m->whole feet: |0.3048-0.4|=0.095 <= max(0.02, 0.1524)); only
        # genuine miscalibration (e.g. an x10 leak) misses, so 1.0 over seeded draws.
        # On main this same instrument measures main's LIVE x10 path (~0.0), so the
        # two-arm delta is the calibration claim itself.
      - name: localize_roundtrip_rate
        direction: max
        threshold: 1.0
        role: guardrail
        bar: floor
        reads_as: "Share of emitted <point> tags (grounded and un-grounded) that parse through the localize stage's exact regex"
        # Baseline-achievable: every tag on main comes from the same pre-existing
        # point_to_molmo_tag / build_qa_pair emit path the regex was built against.
      - name: perspective_flip_accuracy
        direction: max
        threshold: 1.0
        role: diagnostic
        reads_as: "Share of egocentric left/right answers that flip correctly across the 180-degree-yaw second camera (fixture bar 2/2)"
      - name: metric_answer_rel_error
        direction: min
        threshold: 0.05
        role: diagnostic
        reads_as: "Mean relative miss of calibrated answers vs ground truth; typical residual is unit rounding only"
      - name: retired_x10_rel_error
        direction: max
        threshold: 0.5
        role: diagnostic
        reads_as: "Relative error of the retired x10 path on the same scene (paper bar >0.5, expected ~9x) - the intra-run contrast"
      - name: grounded_label_coverage
        direction: max
        threshold: 0.66
        role: diagnostic
        reads_as: "Share of fixture correspondence matches grounded to object labels instead of the anonymous 'feature'"
      - name: inlier_ratio_mean
        direction: max
        threshold: 0.65
        role: diagnostic
        reads_as: "Mean surfaced inlier_ratio over fixture results (0.8 and 0.5 -> 0.65; 0/0 guarded)"
      - name: unusable_extrinsics_accept_rate
        direction: min
        threshold: 0.0
        role: diagnostic
        reads_as: "Share of unusable extrinsics (None, <2 frames, NaN, rotation-only) that parse_extrinsics wrongly accepts -> perspective leak into S=1 data"
      - name: real_registration_spread_ratio
        direction: min
        threshold: 0.05
        role: diagnostic
        bar: goal
        reads_as: "Max cross-frame spread of a landmark's fused world coordinate divided by its metric distance on the real clip (real VGGT S>=2)"
        # Threshold derivation (claim_analysis): registration error propagates ~1:1 into
        # fused distances, so the bar is the PR's own rel_tol=0.05; absolute spread in
        # meters is also printed, ungated.
      - name: real_object_merge_rate
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: goal
        reads_as: "Share of objects masked in >=2 real frames that yield exactly one merged fused cloud"
        # Deterministic union of registered per-frame clouds: any miss is a bug, so 1.0.
      - name: real_depth_conf_variation_rate
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: goal
        reads_as: "Share of real-clip objects with finite retained VGGT depth_conf whose per-scene values are non-constant"
        # The claim asserts depth_conf is retained AND varies per object; both are
        # deterministic properties of the fused output, so 1.0 (std also computed).
      - name: real_scale_sanity_rate
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: goal
        reads_as: "Share of real scenes whose calibrate_metric_scale factor lies within one decade [0.1, 10] (excludes the retired x10 class)"
        # s = median(depthPro_m / vggt_rel) in [0.1, 10] admits VGGT's up-to-scale
        # ambiguity while excluding exactly the retired failure class (factor 10, ~9x).
      - name: real_distance_plausibility_rate
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: goal
        reads_as: "Share of real-clip calibrated distance answers within 50 percent of the DepthPro metric anchor (physically plausible)"
        # |calibrated - depthPro_metric|/depthPro_metric <= 0.5 per emitted answer - the
        # repo test's own retired-failure bar (abs(got-gt) > 0.5*gt) inverted.
      - name: s1_message_hash_crc32
        direction: max
        threshold: 0
        role: diagnostic
        bar: sanity
        reads_as: "Integer CRC-32 over JSON-serialized S=1 messages on identical real frames - parity holds iff both arms print the same integer"
        # Threshold derivation: zlib.crc32 of the payload is an unsigned 32-bit
        # integer, so >= 0 asserts a well-formed fingerprint was emitted (it fails on
        # a missing/None/malformed value). The substantive assertion is CROSS-ARM
        # EQUALITY of the two runs' printed integers (equality of raw hash values
        # cannot be encoded as a single-arm bound), read off the two arms' reports.
      - name: s1_message_count
        direction: max
        threshold: 1
        role: diagnostic
        bar: sanity
        reads_as: "Number of S=1 messages emitted on the real frames (the parity fingerprint is vacuous at zero)"
      - name: fused_new_family_message_count
        direction: max
        threshold: 1
        role: diagnostic
        bar: goal
        reads_as: "Perspective + grounded-correspondence + S>=2-fusion outputs emitted by the fused run on the real frames (absent -> 0 on main)"
compute:
  tier: gpu
  # timeout derivation, ONE arm: apt-get install of open3d's OS runtime libs ~3 min
  # (user_guidance env fix); checkpoint downloads (facebook/VGGT-1B ~5GB,
  # apple/depth-pro-hf ~2GB) ~15 min; VGGT forward on <=3 frames ~2 min; DepthPro
  # <=3 frames ~2 min; CPU synthetic suite <2 min; repair buffer -> 5400 s.
  timeout_s: 5400
variants:
  feature:
    env:
      REMYX_SMOKE: "0"
held_constant:
  - "Synthetic pinhole scene held exactly as tests/test_metric_calibration.py: focal 100 px, depth 5 m, 161x121 grid, two 9x9-px patches (GT size 0.4 m, vertical 0.4 m, horizontal 0.8 m, pairwise mean-NN in [0.632, 0.658] m)"
  - "random.seed(0) and torch.manual_seed(0); frames resized to a fixed 384x288; median-based aggregation -> deterministic outputs"
  - "vggt stub never installed: the real-GPU stage imports vggt unstubbed per user_guidance; no PyPI distribution named vggt exists, so the import is satisfied by the repo's own environment (the PR head's dependency set) and a missing import is handled by the script's declared ImportError fallback (zeroed real-arm metrics), never by a stub or a fabricated number"
  - "open3d runtime OS libs come from apt, never pip: the script's prologue runs apt-get update && apt-get install -y --no-install-recommends libgl1 libegl1 libgomp1 libglib2.0-0 (shell=True, check=True) BEFORE the first open3d/vqasynth import, retrying once with libusb-1.0-0 added if open3d still fails to import; the pip packages: field stays Python-wheels-only"
  - "OpenCV convention (x right, y down, z forward) and the localize <point> regex held as the round-trip contract"
  - "Real clip: >=2 same-scene frames - repo assets/ glob (sorted, first N) else the pinned public fallback (huggingface/documentation-images, revision main, first sorted image, deterministic overlapping crops); the SAME frames feed legacy S=1 and fused S>=2 runs on BOTH arms"
  - "Pinned checkpoints through their own official loaders: facebook/VGGT-1B (vggt.from_pretrained) and apple/depth-pro-hf (transformers AutoModelForDepthEstimation), fetched with HF_TOKEN"
  - "S=1 parity fingerprint: zlib.crc32 over the JSON-serialized S=1 messages (sort_keys=True), printed as the integer s1_message_hash_crc32 on each arm; parity holds iff the two arms print the same integer"
  - "Exactly ONE JSON metrics line, printed by the script's FINAL statement unconditionally on BOTH arms (baseline and PR head) and in REMYX_SMOKE mode; all earlier prints are [remyx]-prefixed debug lines, never JSON, so the metrics line is always the last line of stdout"
avoid:
  - "pytest pass counts as the quality instrument (repo rule: pytest proves code-works only)"
  - "Reimplementing the retired x10 path other than via gen.human_like_distance(10*raw), the idiom the repo's own retired-path test uses"
  - "Guessing signatures: create_point_cloud_from_model, calibrate_metric_scale and compute_object_depth_confidence are not visible in the provided source, so the real-GPU stage binds them by introspecting the live PR-head signatures and calls them for real - never an inline reimplementation or a stubbed stand-in"
  - "Installing OS libraries through pip (libegl1 et al. are apt packages, not wheels) or otherwise changing the dependency channel mid-run"
  - "Wall-clock latency as an instrument on shared compute; the GPU arm reports deterministic geometry-derived quantities"
report:
  headline: "Calibrated answers hit the metric bar on the PR and miss it on main, the real-GPU VGGT+DepthPro path runs unstubbed with sane scale and plausibility, and the S=1 fingerprint matches across arms"
  findings:
    - "{metric_answer_within_bar_rate:%} of emitted distance/size answers land inside the PR's rel_tol=0.05-or-half-unit bar on the pinhole fixture (main's live x10 path is expected near 0)"
    - "{localize_roundtrip_rate:%} of emitted <point> tags parse through localize's exact regex on both arms (guardrail)"
    - "egocentric left/right answers flip correctly across the 180-degree-yaw camera in {perspective_flip_accuracy:%} of fixture prompts, while unusable extrinsics are wrongly accepted at {unusable_extrinsics_accept_rate:%}"
    - "calibrated answers miss ground truth by {metric_answer_rel_error} mean relative error while the retired x10 path sits at {retired_x10_rel_error} (~9x) - the intra-run contrast"
    - "correspondence grounding: {grounded_label_coverage:%} of fixture matches carry object labels (bar 2/3) at mean inlier ratio {inlier_ratio_mean}"
    - "real VGGT S>=2 unstubbed: landmark registration spread ratio {real_registration_spread_ratio}, merged-cloud rate {real_object_merge_rate:%}, depth_conf variation {real_depth_conf_variation_rate:%}"
    - "real DepthPro anchor: scale sanity {real_scale_sanity_rate:%}, distance plausibility {real_distance_plausibility_rate:%}"
    - "S=1 parity: crc32 {s1_message_hash_crc32} over {s1_message_count} messages per arm - parity holds iff the two arms print the same integer"
    - "{fused_new_family_message_count} perspective/grounded/fusion messages emitted by the fused run on the real frames (absent -> 0 on main)"
  establishes:
    - "Real-world calibration of emitted distance/size answers, via {metric_answer_within_bar_rate} on the exact fixture of tests/test_metric_calibration.py, contrasted intra-run by {retired_x10_rel_error}"
    - "The real unstubbed VGGT+DepthPro multi-view path executes and is self-consistent, via {real_registration_spread_ratio}, {real_object_merge_rate}, {real_scale_sanity_rate} and {real_distance_plausibility_rate}"
    - "Single-image-path preservation on identical real clips, via cross-arm equality of {s1_message_hash_crc32}"
  does_not_establish:
    - "Downstream VLM fine-tuning quality or benchmark lift from the new QA families (no model is trained or judged here)"
    - "That perspective QA generalizes beyond the 180-degree-yaw fixture geometry"
  not_measured:
    - "Throughput or latency of the fusion pipeline (wall-clock excluded by design)"
    - "Acceptance rate of the generated data by a downstream filter or human review"
  caveat: "create_point_cloud_from_model / calibrate_metric_scale / compute_object_depth_confidence signatures are bound by introspecting the live PR head, so signature drift surfaces as a crash, not a silent proxy"
  next: "Fine-tune a small VLM on the generated spatial QA and score it on a held-out spatial benchmark to close the loop to model quality"
provenance:
  metric_answer_within_bar_rate: "maintainer_comment:@smellslikm (keep the CPU-observable bars) + repo bar assert_metric_close in tests/test_metric_calibration.py"
  localize_roundtrip_rate: "repo contract LOCALIZE_POINT_RE in tests/test_correspondence.py"
  perspective_flip_accuracy: "repo bar in tests/test_prompts.py (180-degree-yaw flip)"
  metric_answer_rel_error: "inferred from the PR bar rel_tol=0.05"
  retired_x10_rel_error: "repo test test_old_x10_fudge_fails_the_same_bar (abs(got-gt) > 0.5*gt)"
  grounded_label_coverage: "repo fixture in tests/test_correspondence_grounding.py (2 of 3 grounded)"
  inlier_ratio_mean: "repo fixture in tests/test_correspondence_grounding.py (mean of 0.8 and 0.5)"
  unusable_extrinsics_accept_rate: "repo test test_parse_extrinsics_rejects_unusable_inputs"
  real_registration_spread_ratio: "user_guidance (real-GPU arm running VGGT unstubbed at S>=2)"
  real_object_merge_rate: "user_guidance (multi-view fusion mechanism)"
  real_depth_conf_variation_rate: "PR description (depth_conf retained and aggregated per object)"
  real_scale_sanity_rate: "PR description (calibrate_metric_scale with DepthPro metric anchor)"
  real_distance_plausibility_rate: "repo retired-failure bar abs(got-gt) > 0.5*gt inverted"
  s1_message_hash_crc32: "PR description (single-image path preserved byte-for-byte)"
  s1_message_count: "inferred (fingerprint vacuity guard)"
  fused_new_family_message_count: "user_guidance (new-family outputs on the real clip)"
  suite: "repo_runner:eval/eval_vqasynth_spatial_quality.py (reviewer's committed script)"
  compute: "user_guidance (unstubbed VGGT + DepthPro require GPU)"
  held_constant: "protocol_doc:tests/test_metric_calibration.py + tests/test_prompts.py + tests/test_correspondence_grounding.py"
  apt_prologue: "user_guidance (libegl1 is an apt library; install libgl1 libegl1 libgomp1 libglib2.0-0 system-side before importing open3d)"
loop:
  max_iterations: 8
  fix_code: true
```